# 🖼️ Image Captioning using BLIP-2 (Encoder-Decoder Transformer)

## DL Mini Project — COCO Image Caption Dataset

**Problem Statement:** Given an image, automatically generate a natural language caption that describes the content of the image.

**Architecture:** BLIP-2 (Bootstrapping Language-Image Pre-training) — an Encoder-Decoder Transformer with:
- **Vision Encoder (ViT-G/14):** CNN/Transformer backbone that extracts visual features
- **Q-Former (Querying Transformer):** Bridge module with cross-attention between vision and language
- **Language Decoder (OPT-6.7B):** Autoregressive Transformer that generates captions

**Techniques used:** CNN, Transformer, Encoder-Decoder, Attention Mechanism, LoRA Fine-tuning

---

### System Flow Diagram
```
┌───────────┐     ┌──────────────┐     ┌──────────────┐     ┌───────────────┐
│   Input   │────▶│ Vision       │────▶│  Q-Former    │────▶│   Language    │
│   Image   │     │ Encoder      │     │  (Bridge)    │     │   Decoder     │
│ (224×224) │     │ (ViT-G/14)   │     │ Cross-Attn   │     │  (OPT-6.7B)  │
└───────────┘     │ Frozen       │     │ Frozen       │     │  LoRA Tuned   │
                  └──────────────┘     └──────────────┘     └───────┬───────┘
                                                                    │
                                                            ┌───────▼───────┐
                                                            │   Generated   │
                                                            │   Caption     │
                                                            │ "A dog sits   │
                                                            │  on grass"    │
                                                            └───────────────┘
```

## 1. Environment Setup

In [ ]:
%%capture
!pip install -q transformers>=4.41.0 accelerate>=0.30.0 peft>=0.11.0 bitsandbytes>=0.43.0 nltk Pillow matplotlib tqdm pynvml

In [ ]:
import os, json, time, gc, random, glob, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2ForConditionalGeneration, Blip2Processor, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import pynvml
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. GPU Configuration — Maximize H100 Utilization

In [ ]:
# ═══════════════════════════════════════════════════════════════
# H100 GPU OPTIMIZATION — Ensure 100% GPU utilization
# ═══════════════════════════════════════════════════════════════

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Enable TF32 for massive speedup on H100 Tensor Cores
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True  # Auto-tune convolution algorithms

# Set memory allocation strategy for less fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# GPU monitoring utility
pynvml.nvmlInit()
GPU_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def get_gpu_stats():
    util = pynvml.nvmlDeviceGetUtilizationRates(GPU_HANDLE)
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(GPU_HANDLE)
    return {
        'gpu_util': util.gpu,
        'mem_used_gb': mem_info.used / 1024**3,
        'mem_total_gb': mem_info.total / 1024**3,
    }

stats = get_gpu_stats()
print(f"GPU Utilization: {stats['gpu_util']}%")
print(f"VRAM: {stats['mem_used_gb']:.1f}/{stats['mem_total_gb']:.1f} GB")
print(f"TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")

## 3. Dataset Discovery & Exploration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Auto-discover dataset paths (handles various folder structures)
# ═══════════════════════════════════════════════════════════════

BASE_INPUT = '/kaggle/input/datasets/nikhil7280/coco-image-caption'

# Print top-level structure
print("Dataset structure:")
for root, dirs, files in os.walk(BASE_INPUT):
    level = root.replace(BASE_INPUT, '').count(os.sep)
    if level < 3:  # Only show first 3 levels
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level == 2:
            print(f"{indent}  ({len(files)} files)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Locate annotation files and image directories
# ═══════════════════════════════════════════════════════════════

def find_file(base, pattern):
    """Recursively find a file matching pattern."""
    results = glob.glob(os.path.join(base, '**', pattern), recursive=True)
    return results[0] if results else None

# Find annotation files
TRAIN_ANN = find_file(BASE_INPUT, 'captions_train2014.json')
VAL_ANN = find_file(BASE_INPUT, 'captions_val2017.json')
if VAL_ANN is None:
    VAL_ANN = find_file(BASE_INPUT, 'captions_val2014.json')

# Image directories (hardcoded — verified from dataset structure)
TRAIN_IMG_DIR = '/kaggle/input/datasets/nikhil7280/coco-image-caption/train2014/train2014'
VAL_IMG_DIR = '/kaggle/input/datasets/nikhil7280/coco-image-caption/val2017/val2017'

print(f"Train annotations: {TRAIN_ANN}")
print(f"Val annotations:   {VAL_ANN}")
print(f"Train images dir:  {TRAIN_IMG_DIR}")
print(f"Val images dir:    {VAL_IMG_DIR}")

assert TRAIN_ANN and VAL_ANN and TRAIN_IMG_DIR and VAL_IMG_DIR, "Could not find dataset files!"

# Load annotations
with open(TRAIN_ANN, 'r') as f:
    train_data = json.load(f)
with open(VAL_ANN, 'r') as f:
    val_data = json.load(f)

print(f"\nTrain images: {len(train_data['images']):,}")
print(f"Train captions: {len(train_data['annotations']):,}")
print(f"Val images: {len(val_data['images']):,}")
print(f"Val captions: {len(val_data['annotations']):,}")
print(f"Captions per image: {len(train_data['annotations']) / len(train_data['images']):.1f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Visualize sample images with their captions
# ═══════════════════════════════════════════════════════════════

id_to_file = {img['id']: img['file_name'] for img in train_data['images']}
img_to_caps = defaultdict(list)
for ann in train_data['annotations']:
    img_to_caps[ann['image_id']].append(ann['caption'])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
sample_ids = random.sample(list(id_to_file.keys()), 8)

for ax, img_id in zip(axes.flat, sample_ids):
    img_path = os.path.join(TRAIN_IMG_DIR, id_to_file[img_id])
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    cap = img_to_caps[img_id][0][:80] + ('...' if len(img_to_caps[img_id][0]) > 80 else '')
    ax.set_title(cap, fontsize=9, wrap=True)
    ax.axis('off')

plt.suptitle('Sample Training Images with Captions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

del id_to_file, img_to_caps  # Free memory

## 4. Dataset & DataLoader (GPU-Optimized Pipeline)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# High-performance Dataset class
# - Loads images on-the-fly (memory efficient)
# - Preprocesses via BLIP2Processor
# - Uses all 5 captions per image for max accuracy
# ═══════════════════════════════════════════════════════════════

class COCOCaptionDataset(Dataset):
    def __init__(self, image_dir, annotation_file, processor, max_length=128):
        with open(annotation_file, 'r') as f:
            data = json.load(f)
        
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length
        
        # Build image_id -> filename mapping
        self.id_to_filename = {img['id']: img['file_name'] for img in data['images']}
        
        # Build (image_path, caption) pairs — ALL 5 captions per image
        self.samples = []
        skipped = 0
        for ann in data['annotations']:
            img_id = ann['image_id']
            if img_id in self.id_to_filename:
                path = os.path.join(image_dir, self.id_to_filename[img_id])
                if os.path.exists(path):
                    self.samples.append((path, ann['caption']))
                else:
                    skipped += 1
        
        if skipped > 0:
            print(f"Warning: {skipped} samples skipped (missing images)")
        print(f"Dataset initialized: {len(self.samples):,} image-caption pairs")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, caption = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            # Fallback to a random valid sample if image is corrupted
            return self.__getitem__(random.randint(0, len(self) - 1))
        
        encoding = self.processor(
            images=image,
            text=caption,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Squeeze batch dim added by processor
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        
        # Create labels: mask padding with -100 so loss ignores them
        labels = item['input_ids'].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        item['labels'] = labels
        
        return item


def collate_fn(batch):
    """Custom collate to stack variable items into a batch."""
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'input_ids': torch.stack([x['input_ids'] for x in batch]),
        'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch]),
    }

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load processor and create datasets
# ═══════════════════════════════════════════════════════════════

MODEL_NAME = 'Salesforce/blip2-opt-6.7b'
processor = Blip2Processor.from_pretrained(MODEL_NAME)

train_dataset = COCOCaptionDataset(TRAIN_IMG_DIR, TRAIN_ANN, processor, max_length=128)
val_dataset = COCOCaptionDataset(VAL_IMG_DIR, VAL_ANN, processor, max_length=128)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DataLoaders — CRITICAL for 100% GPU utilization
# ═══════════════════════════════════════════════════════════════
# pin_memory=True      → Async CPU→GPU transfer via DMA
# num_workers=4        → Parallel image decode on CPU cores
# prefetch_factor=4    → Buffer 4 batches per worker = 16 total
# persistent_workers   → No worker restart between epochs
# drop_last=True       → Consistent batch sizes for GPU

BATCH_SIZE = 48

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True,
    drop_last=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    collate_fn=collate_fn,
)

print(f"Train batches per epoch: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")
print(f"Effective batch size: {BATCH_SIZE} × 2 (grad accum) = {BATCH_SIZE * 2}")

## 5. Model Loading with LoRA

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load BLIP-2 in bfloat16 (H100 native precision)
# Apply LoRA for parameter-efficient fine-tuning
# ═══════════════════════════════════════════════════════════════

print("Loading BLIP-2 model...")
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

# Freeze all base parameters
for param in model.parameters():
    param.requires_grad = False

# Apply LoRA to language model attention & FFN layers
# Higher rank (r=32) + more targets for maximum accuracy
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2'],
)

model.enable_input_require_grads()  # Required for gradient checkpointing with LoRA
model = get_peft_model(model, peft_config)

# Enable gradient checkpointing to save VRAM → allows larger batches
model.gradient_checkpointing_enable()
model.config.use_cache = False  # Incompatible with gradient checkpointing

model.print_trainable_parameters()

stats = get_gpu_stats()
print(f"\nVRAM after model load: {stats['mem_used_gb']:.1f}/{stats['mem_total_gb']:.1f} GB")

## 6. Training Loop — Full H100 Utilization

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Training hyperparameters
# ═══════════════════════════════════════════════════════════════

NUM_EPOCHS = 3
GRAD_ACCUM_STEPS = 2          # Effective batch = 32 × 2 = 64
LEARNING_RATE = 1e-4
WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0
LOG_EVERY = 100
SAVE_EVERY = 2000
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

total_steps = len(train_loader) * NUM_EPOCHS
optimizer_steps = total_steps // GRAD_ACCUM_STEPS

# Optimizer — only LoRA params are trainable
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=0.05,
    betas=(0.9, 0.999),
)

# Cosine schedule with warmup
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=optimizer_steps,
)

print(f"Total forward steps: {total_steps:,}")
print(f"Optimizer steps: {optimizer_steps:,}")
print(f"Estimated time: ~{total_steps * 0.4 / 3600:.1f} hours")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRAINING LOOP
# Key optimizations for 100% GPU utilization:
# 1. non_blocking=True on all .to(device) calls
# 2. bfloat16 autocast (H100 native)
# 3. set_to_none=True for zero_grad (faster than filling zeros)
# 4. Gradient accumulation for large effective batch
# 5. DataLoader prefetching hides I/O latency
# ═══════════════════════════════════════════════════════════════

train_losses = []
gpu_utils = []
lr_history = []
best_val_loss = float('inf')
global_step = 0
training_start = time.time()

print("="*70)
print("STARTING TRAINING")
print("="*70)

model.train()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_loss = 0.0
    num_batches = 0
    
    for step, batch in enumerate(train_loader):
        # ── Non-blocking GPU transfer (overlap with compute) ──
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        
        # ── Forward pass with bfloat16 autocast (H100 native) ──
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / GRAD_ACCUM_STEPS
        
        # ── Backward ──
        loss.backward()
        
        # ── Optimizer step (every GRAD_ACCUM_STEPS) ──
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                MAX_GRAD_NORM
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)  # Faster than zero filling
        
        current_loss = loss.item() * GRAD_ACCUM_STEPS
        epoch_loss += current_loss
        num_batches += 1
        global_step += 1
        
        # ── Logging ──
        if global_step % LOG_EVERY == 0:
            stats = get_gpu_stats()
            avg_loss = epoch_loss / num_batches
            elapsed = time.time() - training_start
            steps_per_sec = global_step / elapsed
            eta_hours = (total_steps - global_step) / steps_per_sec / 3600
            
            train_losses.append(avg_loss)
            gpu_utils.append(stats['gpu_util'])
            lr_history.append(scheduler.get_last_lr()[0])
            
            print(
                f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                f"Step {step+1}/{len(train_loader)} | "
                f"Loss: {current_loss:.4f} (avg: {avg_loss:.4f}) | "
                f"LR: {scheduler.get_last_lr()[0]:.2e} | "
                f"GPU: {stats['gpu_util']}% | "
                f"VRAM: {stats['mem_used_gb']:.1f}GB | "
                f"Speed: {steps_per_sec:.1f} steps/s | "
                f"ETA: {eta_hours:.1f}h"
            )
        
        # ── Checkpoint saving ──
        if global_step % SAVE_EVERY == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f'step_{global_step}')
            model.save_pretrained(ckpt_path)
            print(f"  ✓ Checkpoint saved: {ckpt_path}")
    
    # ── Epoch summary ──
    epoch_time = time.time() - epoch_start
    avg_epoch_loss = epoch_loss / num_batches
    print(f"\n{'─'*70}")
    print(f"Epoch {epoch+1} complete | Avg Loss: {avg_epoch_loss:.4f} | Time: {epoch_time/60:.1f} min")
    print(f"{'─'*70}\n")
    
    # Save epoch checkpoint
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch+1}')
    model.save_pretrained(ckpt_path)
    processor.save_pretrained(ckpt_path)

total_time = time.time() - training_start
print(f"\n{'='*70}")
print(f"TRAINING COMPLETE — Total time: {total_time/3600:.2f} hours")
print(f"{'='*70}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Plot training curves
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, color='#FF6B6B', linewidth=2)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Steps (×100)')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(gpu_utils, color='#4ECDC4', linewidth=2)
axes[1].set_title('GPU Utilization (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Steps (×100)')
axes[1].set_ylabel('GPU %')
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.3)

axes[2].plot(lr_history, color='#45B7D1', linewidth=2)
axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Steps (×100)')
axes[2].set_ylabel('LR')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Training Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation — BLEU Scores

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Evaluate on validation set
# Compute BLEU-1, BLEU-2, BLEU-3, BLEU-4 scores
# ═══════════════════════════════════════════════════════════════

# Build image_id → list of reference captions for val set
with open(VAL_ANN, 'r') as f:
    val_anns = json.load(f)

val_id_to_file = {img['id']: img['file_name'] for img in val_anns['images']}
val_img_refs = defaultdict(list)
for ann in val_anns['annotations']:
    val_img_refs[ann['image_id']].append(ann['caption'].lower().strip())

# Evaluate on a subset for speed (5000 images)
eval_image_ids = list(val_id_to_file.keys())[:5000]

model.eval()
all_references = []
all_hypotheses = []

print(f"Evaluating on {len(eval_image_ids)} validation images...")
eval_start = time.time()

EVAL_BATCH = 32
for i in tqdm(range(0, len(eval_image_ids), EVAL_BATCH), desc='Evaluating'):
    batch_ids = eval_image_ids[i:i+EVAL_BATCH]
    images = []
    batch_refs = []
    
    for img_id in batch_ids:
        img_path = os.path.join(VAL_IMG_DIR, val_id_to_file[img_id])
        try:
            img = Image.open(img_path).convert('RGB')
            images.append(img)
            # References: list of tokenized reference captions
            refs = [nltk.word_tokenize(ref) for ref in val_img_refs[img_id]]
            batch_refs.append(refs)
        except Exception:
            continue
    
    if not images:
        continue
    
    inputs = processor(images=images, return_tensors='pt', padding=True)
    pixel_values = inputs['pixel_values'].to(device, non_blocking=True)
    
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        generated_ids = model.generate(
            pixel_values=pixel_values,
            max_new_tokens=50,
            num_beams=5,
            early_stopping=True,
        )
    
    generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
    
    for text, refs in zip(generated_texts, batch_refs):
        hyp = nltk.word_tokenize(text.lower().strip())
        all_hypotheses.append(hyp)
        all_references.append(refs)

eval_time = time.time() - eval_start
print(f"Evaluation time: {eval_time/60:.1f} minutes")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Compute BLEU scores
# ═══════════════════════════════════════════════════════════════

smooth = SmoothingFunction().method1

bleu_1 = corpus_bleu(all_references, all_hypotheses, weights=(1, 0, 0, 0), smoothing_function=smooth)
bleu_2 = corpus_bleu(all_references, all_hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
bleu_3 = corpus_bleu(all_references, all_hypotheses, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
bleu_4 = corpus_bleu(all_references, all_hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

print("\n" + "="*50)
print("        EVALUATION RESULTS")
print("="*50)
print(f"  BLEU-1: {bleu_1:.4f}")
print(f"  BLEU-2: {bleu_2:.4f}")
print(f"  BLEU-3: {bleu_3:.4f}")
print(f"  BLEU-4: {bleu_4:.4f}")
print(f"  Images evaluated: {len(all_hypotheses)}")
print("="*50)

# Save results
results = {
    'bleu_1': bleu_1, 'bleu_2': bleu_2, 'bleu_3': bleu_3, 'bleu_4': bleu_4,
    'num_eval_images': len(all_hypotheses),
    'training_time_hours': total_time / 3600,
    'model': MODEL_NAME, 'lora_r': 32, 'epochs': NUM_EPOCHS, 'batch_size': BATCH_SIZE,
}
with open('/kaggle/working/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to eval_results.json")

## 8. Qualitative Results — Sample Predictions

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Show predictions vs ground truth on random val images
# ═══════════════════════════════════════════════════════════════

model.eval()
sample_ids = random.sample(list(val_id_to_file.keys()), 8)

fig, axes = plt.subplots(2, 4, figsize=(24, 12))

for ax, img_id in zip(axes.flat, sample_ids):
    img_path = os.path.join(VAL_IMG_DIR, val_id_to_file[img_id])
    image = Image.open(img_path).convert('RGB')
    
    inputs = processor(images=image, return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(device, non_blocking=True)
    
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        gen_ids = model.generate(pixel_values=pixel_values, max_new_tokens=50, num_beams=5)
    
    pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    gt = val_img_refs[img_id][0][:80]
    
    ax.imshow(image)
    ax.set_title(f'Pred: {pred}\n\nGT: {gt}', fontsize=9, color='darkgreen', wrap=True)
    ax.axis('off')

plt.suptitle('Model Predictions vs Ground Truth', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Final Model

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Save LoRA adapters + processor for local inference UI
# Download from Kaggle Output after notebook completes
# ═══════════════════════════════════════════════════════════════

SAVE_DIR = '/kaggle/working/blip2-coco-captioner'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save LoRA adapter weights
model.save_pretrained(SAVE_DIR)
print(f"LoRA adapters saved to {SAVE_DIR}")

# Save processor
processor.save_pretrained(SAVE_DIR)
print(f"Processor saved to {SAVE_DIR}")

# Save model config for the UI app
config = {
    'base_model': MODEL_NAME,
    'lora_r': 32,
    'lora_alpha': 64,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2'],
    'bleu_4': bleu_4,
    'training_epochs': NUM_EPOCHS,
    'training_time_hours': total_time / 3600,
}
with open(os.path.join(SAVE_DIR, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

# List saved files and sizes
print(f"\nSaved files:")
total_size = 0
for root, _, files in os.walk(SAVE_DIR):
    for fname in files:
        fpath = os.path.join(root, fname)
        size = os.path.getsize(fpath)
        total_size += size
        print(f"  {fname}: {size / 1024**2:.1f} MB")
print(f"\nTotal model size: {total_size / 1024**2:.1f} MB")
print("\n✅ Download the 'blip2-coco-captioner' folder from Kaggle Output to run the local UI!")

## Results Summary

### Architecture
- **Model:** BLIP-2 (ViT-G/14 + Q-Former + OPT-6.7B)
- **Fine-tuning:** LoRA (r=32, α=64) on language model
- **Trainable params:** ~31M / ~3.8B total (~0.8%)

### Training
- **Dataset:** COCO 2014 (~414K image-caption pairs)
- **GPU:** NVIDIA H100 (bfloat16, TF32 tensor cores)
- **Batch size:** 32 (effective 64 with gradient accumulation)

### Key Techniques
| Component | Role |
|-----------|------|
| ViT-G/14 (Vision Transformer) | Image feature extraction (Encoder) |
| Q-Former | Vision-language alignment via cross-attention |
| OPT-6.7B (Transformer) | Caption generation (Decoder) |
| LoRA | Parameter-efficient fine-tuning |
| Beam Search | Improved caption quality at inference |